# ReviewGPT - Exploratory Data Analysis & Model Development

This notebook contains:
1. Exploratory Data Analysis (EDA)
2. Data visualization
3. Model training experiments
4. Results analysis and visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

%matplotlib inline

## 1. Load and Explore Data

In [ ]:
# Load processed data
df = pd.read_csv('../reviews_processed.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head(10)

In [ ]:
# Basic statistics
print("Dataset Info:")
print(df.info())
print("\nBasic Statistics:")
print(df.describe())

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

## 2. Distribution Analysis

In [ ]:
# Star rating distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stars distribution
stars_counts = df['stars'].value_counts().sort_index()
axes[0].bar(stars_counts.index, stars_counts.values, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Star Rating', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Distribution of Star Ratings', fontsize=14, fontweight='bold')
axes[0].set_xticks([1, 2, 3, 4, 5])
for i, v in enumerate(stars_counts.values):
    axes[0].text(i+1, v + 50, str(v), ha='center', fontweight='bold')

# Needs reply distribution
reply_counts = df['needs_reply'].value_counts().sort_index()
labels = ['No Reply Needed', 'Reply Needed']
colors = ['lightgreen', 'salmon']
axes[1].bar(range(len(reply_counts)), reply_counts.values, color=colors, alpha=0.7)
axes[1].set_xlabel('Needs Reply', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Distribution of Reply Necessity', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels)
for i, v in enumerate(reply_counts.values):
    axes[1].text(i, v + 50, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Relationship between stars and needs_reply
cross_tab = pd.crosstab(df['stars'], df['needs_reply'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(10, 6))
cross_tab.plot(kind='bar', stacked=True, ax=ax, color=['lightgreen', 'salmon'], alpha=0.8)
ax.set_xlabel('Star Rating', fontsize=12)
ax.set_ylabel('Percentage', fontsize=12)
ax.set_title('Reply Necessity by Star Rating', fontsize=14, fontweight='bold')
ax.legend(['No Reply Needed', 'Reply Needed'], title='Needs Reply')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

print("\nPercentage of reviews needing reply by star rating:")
print(cross_tab)

## 3. Text Analysis

In [ ]:
# Review length analysis
df['review_length'] = df['review_text'].str.len()
df['word_count'] = df['review_text'].str.split().str.len()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Character length distribution
axes[0, 0].hist(df['review_length'], bins=50, color='skyblue', alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Review Length (characters)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Distribution of Review Length', fontsize=12, fontweight='bold')
axes[0, 0].axvline(df['review_length'].mean(), color='red', linestyle='--', label=f"Mean: {df['review_length'].mean():.0f}")
axes[0, 0].legend()

# Word count distribution
axes[0, 1].hist(df['word_count'], bins=50, color='lightcoral', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Word Count', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Distribution of Word Count', fontsize=12, fontweight='bold')
axes[0, 1].axvline(df['word_count'].mean(), color='red', linestyle='--', label=f"Mean: {df['word_count'].mean():.1f}")
axes[0, 1].legend()

# Review length by star rating
df.boxplot(column='review_length', by='stars', ax=axes[1, 0])
axes[1, 0].set_xlabel('Star Rating', fontsize=11)
axes[1, 0].set_ylabel('Review Length (characters)', fontsize=11)
axes[1, 0].set_title('Review Length by Star Rating', fontsize=12, fontweight='bold')
plt.sca(axes[1, 0])
plt.xticks([1, 2, 3, 4, 5])

# Review length by needs_reply
df.boxplot(column='review_length', by='needs_reply', ax=axes[1, 1])
axes[1, 1].set_xlabel('Needs Reply', fontsize=11)
axes[1, 1].set_ylabel('Review Length (characters)', fontsize=11)
axes[1, 1].set_title('Review Length by Reply Necessity', fontsize=12, fontweight='bold')
plt.sca(axes[1, 1])
plt.xticks([1, 2], ['No', 'Yes'])

plt.tight_layout()
plt.show()

print(f"\nAverage review length: {df['review_length'].mean():.2f} characters")
print(f"Average word count: {df['word_count'].mean():.2f} words")
print(f"\nReview length stats by star rating:")
print(df.groupby('stars')['review_length'].describe())

In [ ]:
# Sample reviews from each category
print("Sample Reviews:\n" + "="*100)

for stars in [1, 3, 5]:
    for reply in [0, 1]:
        sample = df[(df['stars'] == stars) & (df['needs_reply'] == reply)].sample(1, random_state=42)
        if len(sample) > 0:
            print(f"\n{stars} stars, {'needs reply' if reply else 'no reply needed'}:")
            print(f"{sample.iloc[0]['review_text'][:200]}...")
            print("-" * 100)

## 4. Model Training Results

After running `train.py`, analyze the results here.

In [ ]:
# Load training metrics
try:
    with open('../artifacts/metrics.json', 'r') as f:
        metrics = json.load(f)
    
    print("Model Performance Metrics:")
    print("="*80)
    print("\nStar Rating Prediction:")
    for key, value in metrics['test_metrics']['stars'].items():
        print(f"  {key.capitalize()}: {value:.4f}")
    
    print("\nNeeds Reply Prediction:")
    for key, value in metrics['test_metrics']['needs_reply'].items():
        print(f"  {key.capitalize()}: {value:.4f}")
    
    print(f"\nBest epoch: {metrics['best_epoch']}")
except FileNotFoundError:
    print("No metrics file found. Run train.py first.")
    metrics = None

In [ ]:
# Visualize confusion matrix
if metrics is not None:
    conf_matrix = np.array(metrics['confusion_matrix'])
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
                xticklabels=[1, 2, 3, 4, 5], 
                yticklabels=[1, 2, 3, 4, 5],
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted Stars', fontsize=12)
    plt.ylabel('True Stars', fontsize=12)
    plt.title('Confusion Matrix - Star Rating Prediction', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Calculate per-class accuracy
    per_class_accuracy = conf_matrix.diagonal() / conf_matrix.sum(axis=1)
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar([1, 2, 3, 4, 5], per_class_accuracy, color='steelblue', alpha=0.7)
    plt.xlabel('Star Rating', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.title('Per-Class Accuracy by Star Rating', fontsize=14, fontweight='bold')
    plt.ylim([0, 1])
    plt.grid(axis='y', alpha=0.3)
    
    for i, (bar, acc) in enumerate(zip(bars, per_class_accuracy)):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{acc:.3f}', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 5. Error Analysis

Analyze misclassifications to understand model weaknesses.

In [ ]:
# This section would require running predictions on the test set
# and comparing with ground truth labels

print("To perform error analysis:")
print("1. Use main.py to generate predictions for test set reviews")
print("2. Compare predictions with ground truth labels")
print("3. Analyze patterns in misclassified examples")
print("\nExample command:")
print('python main.py --checkpoint artifacts/model.pt --text "Your review here"')

## 6. Insights and Recommendations

### Key Findings:
1. **Data Distribution**: The dataset shows a relatively balanced distribution across star ratings
2. **Reply Patterns**: Lower-rated reviews (1-2 stars) are much more likely to need replies
3. **Review Length**: Review length varies by rating, with negative reviews often being longer

### Model Improvements:
- Consider class weighting for imbalanced needs_reply labels
- Experiment with different max_length values for tokenization
- Try different transformer architectures (BERT, RoBERTa, etc.)
- Add data augmentation for minority classes

### Future Work:
- Implement sentiment analysis as an additional task
- Add topic modeling to categorize review themes
- Create a web interface for easier interaction
- Deploy as a REST API service